In [ ]:
FILES={'hybrid_inputs.py': '"""CPU preflight for hybrid retrieval: part discovery, coverage and row alignment."""\nimport hashlib\nimport json\nimport math\nfrom collections import Counter\nfrom pathlib import Path\n\nEMBEDDING_SPECS = {\n    \'qwen4b\': dict(model=\'Qwen/Qwen3-Embedding-4B\', dim=2560, doc_prefix=\'\', directory=\'Qwen3-Embedding-4B\'),\n    \'qwen8b\': dict(model=\'Qwen/Qwen3-Embedding-8B\', dim=4096, doc_prefix=\'\', directory=\'Qwen3-Embedding-8B\'),\n    \'octen8b\': dict(model=\'Octen/Octen-Embedding-8B\', dim=4096, doc_prefix=\' \', directory=\'Octen-Embedding-8B\'),\n}\n\n\ndef embedding_weights(selected, fusion, weights):\n    """Normalized model weights: zero-weight models need no attached assets."""\n    if not fusion:\n        assert selected in EMBEDDING_SPECS, f\'Unknown EMBEDDING_MODEL: {selected}\'\n        return {selected: 1.0}\n    assert set(weights) == set(EMBEDDING_SPECS), \'MODEL_WEIGHTS must specify qwen4b, qwen8b, octen8b\'\n    assert all(isinstance(w, (int, float)) and not isinstance(w, bool) and math.isfinite(w) and w >= 0\n               for w in weights.values()), \'Model weights must be finite and nonnegative\'\n    total = sum(weights.values())\n    assert math.isfinite(total) and total > 0, \'At least one model weight must be positive\'\n    return {k: weights[k] / total for k in EMBEDDING_SPECS if weights[k] > 0}\n\n\ndef group_embedding_vectors(paths, active):\n    """Resolve model identity from its manifest, never from dimension or part filename."""\n    import re\n    by_repo = {EMBEDDING_SPECS[k][\'model\']: k for k in active}\n    found = {k: [] for k in active}\n    manifests = {}\n    for raw in paths:\n        path = Path(raw)\n        if not re.fullmatch(r\'(vi|en|zh)-part-\\d+\', path.stem):\n            continue\n        manifest_path = path.parent.parent / \'embed_manifest.json\'\n        if manifest_path not in manifests:\n            manifests[manifest_path] = json.loads(manifest_path.read_text(encoding=\'utf-8\'))\n        key = by_repo.get(manifests[manifest_path].get(\'model\'))\n        if key is not None:\n            found[key].append(str(path))\n    for key in active:\n        assert found[key], f\'Missing corpus vectors for {key}; attach its three baseline sources and additions dataset\'\n    return found\n\n\ndef validate_query_cache(saved, vectors, key, variants, task, query_ids, legacy=False, revision=None):\n    import numpy as np\n    assert saved[\'variants\'] == variants and saved[\'task\'] == task, \'Cached query variants/instruction differ\'\n    if legacy:\n        assert key == \'qwen4b\', \'Unlabelled legacy query vectors are supported only for qwen4b\'\n        assert saved.get(\'model\', EMBEDDING_SPECS[key][\'model\']) == EMBEDDING_SPECS[key][\'model\']\n    else:\n        assert saved[\'model\'] == EMBEDDING_SPECS[key][\'model\'], \'Cached query model differs\'\n        assert saved[\'query_ids\'] == list(query_ids), \'Cached query ID order differs\'\n        assert saved[\'query_format\'] == \'Instruct: {task}\\nQuery:{text}\' and saved[\'max_length\'] == 1024, \'Cached query encoding contract differs\'\n        assert revision is None or saved.get(\'model_revision\') == revision, \'Cached model revision differs\'\n    assert vectors.shape == (len(variants), EMBEDDING_SPECS[key][\'dim\']) and np.isfinite(vectors).all(), \'Invalid cached query vectors\'\n    assert np.max(abs(np.linalg.norm(vectors.astype(np.float32), axis=1) - 1)) < .01, \'Cached queries must have unit norm\'\n\n\ndef unique_parts(paths, chunk=False):\n    result = {}\n    for raw in paths:\n        path = Path(raw)\n        key = f\'{path.parent.name}-{path.stem}\' if chunk else path.stem\n        assert key.split(\'-\')[0] in {\'vi\', \'en\', \'zh\'}, f\'Unknown language in part: {path}\'\n        assert key not in result, f\'Duplicate part {key}: {result.get(key)} and {path}\'\n        result[key] = str(path)\n    assert result, \'No chunk parts found\' if chunk else \'No vector parts found\'\n    return result\n\n\ndef select_input_parts(chunk_files, vector_files, langs):\n    langs = list(langs)\n    assert langs and len(set(langs)) == len(langs) and set(langs) <= {\'vi\', \'en\', \'zh\'}, f\'Invalid LANGS: {langs}\'\n    chunks = unique_parts(chunk_files, chunk=True)\n    vectors = unique_parts(vector_files)\n    chunks = {k: p for k, p in chunks.items() if k.split(\'-\')[0] in langs}\n    vectors = {k: p for k, p in vectors.items() if k.split(\'-\')[0] in langs}\n    assert {k.split(\'-\')[0] for k in chunks} == set(langs), f\'Missing chunk languages: {langs}\'\n    assert set(chunks) == set(vectors), (\n        f\'Chunk/vector parts differ: missing vectors={sorted(set(chunks)-set(vectors))}; \'\n        f\'extra vectors={sorted(set(vectors)-set(chunks))}\')\n    return chunks, vectors\n\n\ndef json_fingerprint(path):\n    path = Path(path)\n    return dict(path=str(path), sha256=hashlib.sha256(path.read_bytes()).hexdigest())\n\n\ndef validate_hybrid_inputs(chunk_files, vector_files, langs, manifest_path, expected_version=\'v2\', progress=None, model_key=\'qwen4b\'):\n    import numpy as np\n    import pyarrow.parquet as pq\n\n    spec = EMBEDDING_SPECS[model_key]\n    manifest_path = Path(manifest_path)\n    manifest = json.loads(manifest_path.read_text(encoding=\'utf-8\'))\n    assert not expected_version or manifest[\'dataset_version\'] == expected_version, (\n        f\'Expected corpus {expected_version}, found {manifest.get("dataset_version")}\')\n    chunks, vectors = select_input_parts(chunk_files, vector_files, langs)\n    schemas = {}; rows = Counter(); seen = set(); vector_manifests = {}\n    for key in sorted(vectors):\n        path = Path(vectors[key]); chunk_path = Path(chunks[key]); lang = key.split(\'-\')[0]\n        assert chunk_path.resolve().is_relative_to(manifest_path.parent.resolve()), f\'Chunk part outside corpus root: {chunk_path}\'\n        embed_manifest_path = path.parent.parent / \'embed_manifest.json\'\n        if str(embed_manifest_path) not in vector_manifests:\n            m = json.loads(embed_manifest_path.read_text(encoding=\'utf-8\'))\n            assert m[\'complete\'] and m[\'model\'] == spec[\'model\'] and m[\'dim\'] == spec[\'dim\'], f\'Incompatible vector manifest: {embed_manifest_path}\'\n            assert m.get(\'doc_prefix\', \'\') == spec[\'doc_prefix\'], f\'Incompatible document prefix: {embed_manifest_path}\'\n            assert m[\'max_length\'] == 1024 and m[\'dtype\'] == \'torch.bfloat16\', f\'Incompatible embedding contract: {embed_manifest_path}\'\n            for field, expected in [(\'instruction\', \'\'), (\'pooling\', \'last_token\'), (\'padding_side\', \'left\'), (\'normalize\', True)]:\n                assert m.get(field, expected) == expected, f\'Incompatible {field}: {embed_manifest_path}\'\n            vector_manifests[str(embed_manifest_path)] = dict(**json_fingerprint(embed_manifest_path), model=m[\'model\'], dim=m[\'dim\'],\n                model_revision=m.get(\'model_revision\'), output_dtype=m.get(\'output_dtype\', \'float16\'))\n        array = np.load(path, mmap_mode=\'r\', allow_pickle=False)\n        assert array.ndim == 2 and array.dtype == np.float16 and array.shape[1] == spec[\'dim\'], (key, array.dtype, array.shape)\n        ids_path = path.with_name(f\'{key}-ids.parquet\')\n        ids_table = pq.read_table(ids_path, columns=[\'chunk_id\'])\n        chunk_table = pq.read_table(chunk_path, columns=[\'chunk_id\', \'lang\'])\n        assert ids_table[\'chunk_id\'].null_count == chunk_table[\'chunk_id\'].null_count == 0, f\'Null ID in {key}\'\n        ids = ids_table[\'chunk_id\'].to_numpy(); expected_ids = chunk_table[\'chunk_id\'].to_numpy()\n        assert array.shape[0] == len(ids) == len(expected_ids) and len(ids) > 0, f\'Row count mismatch: {key}\'\n        assert np.array_equal(ids, expected_ids), f\'Chunk ID order mismatch: {key}\'\n        assert set(chunk_table[\'lang\'].to_pylist()) == {lang}, f\'Wrong language rows: {key}\'\n        part_ids = set(ids.tolist())\n        assert len(part_ids) == len(ids) and not seen.intersection(part_ids), f\'Duplicate chunk IDs: {key}\'\n        seen.update(part_ids); rows[lang] += len(ids)\n        schemas[key] = dict(chunk_path=str(chunk_path), vector_path=str(path), ids_path=str(ids_path), rows=len(ids), language=lang)\n        del array, ids, expected_ids, ids_table, chunk_table, part_ids\n        if progress: progress(f\'preflight {key}: {schemas[key]["rows"]:,} aligned IDs\')\n    for lang in langs:\n        assert rows[lang] == manifest[\'languages\'][lang][\'chunks\'], f\'Corpus manifest count mismatch for {lang}: {rows[lang]}\'\n    provenance = dict(model_key=model_key, model=spec[\'model\'], dim=spec[\'dim\'], corpus_version=manifest[\'dataset_version\'], corpus_manifest=json_fingerprint(manifest_path),\n        selected_languages=list(langs), chunk_count=sum(rows.values()), vector_count=sum(rows.values()),\n        rows_by_language=dict(rows), part_count=len(schemas), parts=schemas,\n        vector_manifests=list(vector_manifests.values()), all_part_ids_aligned=True)\n    return chunks, vectors, provenance\n', 'hybrid_retrieval.py': '"""CPU-side logic of the hybrid retrieval notebook (scripts/build_hybrid_notebook.py copies this file into it).\n\nQuery variants: rule-based cleanup of patient chit-chat and a split of long queries into sentences and\ncomma clauses; no LLM. Every variant is searched on its own and the lists are fused with weighted RRF.\nDocument ranking: a document scores the max, or the sum of its 3 best chunk scores (DOC_AGG). Scores\nonly depend on the submitted sets, so the ranking matters through what falls inside N_DOCS / N_CHUNKS.\n\nWith one dense list, DOC_AGG=\'max\' and no reranker, assemble() gives exactly the old d200-c150 lists\n(scripts/retrieve_local_4b.py): docs in order of their best chunk, chunk docs appended when missing.\n"""\nimport re\nimport unicodedata\n\nCHITCHAT = re.compile(\n    r\'(?i)\\b(?:\'\n    r\'(?:xin )?chào (?:bác s[ĩỹi]|bs)[^,.?!]*[,.!]?\'\n    r\'|(?:cho|xin cho) (?:em|tôi|mình|cháu|con|chị|anh) (?:được )?hỏi(?: (?:bác s[ĩỹi]|bs))?(?: là)?\'\n    r\'|(?:em |tôi |mình )?xin hỏi\'\n    r\'|bác s[ĩỹi] ơi\'\n    r\'|(?:em |tôi |mình |xin )?(?:chân thành )?cảm ơn(?: (?:bác s[ĩỹi]|bs))?(?: nhiều)?(?: ạ)?\'\n    r\'|mong (?:bác s[ĩỹi]|bs)[^.?!]*(?:tư vấn|giải đáp|trả lời|giúp)[^.?!]*\'\n    r\'|nhờ (?:bác s[ĩỹi]|bs)[^.?!]*(?:tư vấn|giải đáp)[^.?!]*\'\n    r\')\')\nSENTENCE = re.compile(r\'(?<=[.?!;…])\\s+|\\n+\')\n\n\ndef clean_query(text):\n    """Drop greetings, thanks and \'cho em hỏi\'-style openers; keep every medical word."""\n    text = CHITCHAT.sub(\' \', unicodedata.normalize(\'NFC\', text))\n    return re.sub(r\'\\s+\', \' \', text).strip(\' ,.\')\n\n\ndef sub_queries(query, split_min_words=20, min_sub_words=4, max_subq=4):\n    """Sentences of a long query, plus the comma clauses of a sentence when every clause has 3+ words.\n\n    Short queries (most are one sentence of ~18 words) return []: they are searched whole only.\n    """\n    if len(query.split()) < split_min_words:\n        return []\n    found = []\n    for sentence in SENTENCE.split(clean_query(query)):\n        sentence = sentence.strip(\' ,.;\')\n        if len(sentence.split()) < min_sub_words:\n            continue\n        found.append(sentence)\n        clauses = [part.strip() for part in sentence.split(\',\')]\n        if len(clauses) >= 2 and all(len(part.split()) >= 3 for part in clauses):\n            found.extend(clauses)\n    seen, out = {query.lower().strip()}, []\n    for text in found:\n        if text.lower() not in seen:\n            seen.add(text.lower())\n            out.append(text)\n    return out[:max_subq]\n\n\ndef rrf(lists, k=60):\n    """Weighted reciprocal-rank fusion. lists: [(ranked ids, weight)]. Returns [(id, score)], best first.\n\n    Ties keep the order of first appearance, so a single list comes back in its own order.\n    """\n    score, first = {}, {}\n    for ids, weight in lists:\n        for rank, cid in enumerate(ids):\n            score[cid] = score.get(cid, 0.0) + weight / (k + rank + 1)\n            first.setdefault(cid, len(first))\n    return sorted(score.items(), key=lambda item: (-item[1], first[item[0]]))\n\n\ndef model_candidate_lists(results, weights, qi, sub_weight, prf_weight):\n    """One RRF stage: model weights multiply full/sub/PRF lists; BM25 is added once by caller."""\n    lists = []\n    for key, weight in weights.items():\n        result = results[key]\n        lists.append((list(result[\'full_ids\'][qi]), weight))\n        lists.extend((list(ids), weight * sub_weight) for ids in result[\'sub_lists\'][qi])\n        if result[\'prf_ids\'] is not None:\n            lists.append((list(result[\'prf_ids\'][qi]), weight * prf_weight))\n    return lists\n\n\ndef doc_ranking(scored, docs_of, agg=\'max\'):\n    """Documents of the scored chunks ([(chunk id, score)], best first), best document first.\n\n    agg=\'max\': a document scores its best chunk (the old ranking). agg=\'top3\': the sum of its 3 best chunks,\n    so a document with several matching chunks beats one with a single lucky match.\n    """\n    per_doc, first = {}, {}\n    for cid, score in scored:\n        for doc_id in docs_of(cid):\n            per_doc.setdefault(doc_id, []).append(score)\n            first.setdefault(doc_id, len(first))\n    if agg == \'max\':\n        value = {doc: max(scores) for doc, scores in per_doc.items()}\n    elif agg == \'top3\':\n        value = {doc: sum(sorted(scores, reverse=True)[:3]) for doc, scores in per_doc.items()}\n    else:\n        raise ValueError(f\'unknown DOC_AGG {agg!r}\')\n    return sorted(per_doc, key=lambda doc: (-value[doc], first[doc]))\n\n\ndef assemble(scored, fill_ids, docs_of, n_docs, n_chunks, agg=\'max\'):\n    """One query\'s (relevant_docs, [(doc_id, chunk id)]).\n\n    scored: [(chunk id, score)] best first, the chunks with a final score (the reranked pool, or all fused\n    candidates). fill_ids: further candidate chunk ids in fused order, used only to top the document list\n    up to n_docs. Chunk documents missing from the list are appended, as in the earlier submissions.\n    """\n    docs = doc_ranking(scored, docs_of, agg)[:n_docs]\n    listed = set(docs)\n    for cid in fill_ids:\n        if len(docs) >= n_docs:\n            break\n        for doc_id in docs_of(cid):\n            if doc_id not in listed and len(docs) < n_docs:\n                listed.add(doc_id)\n                docs.append(doc_id)\n    chunks = []\n    for cid, _ in scored[:n_chunks]:\n        for doc_id in docs_of(cid):\n            chunks.append((doc_id, cid))\n            if doc_id not in listed:\n                listed.add(doc_id)\n                docs.append(doc_id)\n    return docs, chunks\n', 'hybrid_embedding_cells.py': '"""Embedding/search cells used by build_hybrid_notebook.py. Edit here and regenerate."""\n\nENCODE = r\'\'\'\nstarted = time.perf_counter()\nQUERY_VECTORS = {}\nfor model_key in ACTIVE_MODEL_WEIGHTS:\n    spec = EMBEDDING_SPECS[model_key]\n    metadata = dict(model=spec[\'model\'], model_key=model_key, task=TASK, variants=VARIANTS,\n                    query_ids=QUERY_IDS, model_revision=MODEL_REVISIONS[model_key],\n                    query_format=\'Instruct: {task}\\nQuery:{text}\', max_length=1024)\n    if ENCODE_HERE:\n        from transformers import AutoModel, AutoTokenizer\n        import torch.nn.functional as F\n        os.environ[\'HF_HUB_OFFLINE\'] = \'1\'\n        tok = AutoTokenizer.from_pretrained(str(MODEL_PATHS[model_key]), local_files_only=True)\n        tok.padding_side = \'left\'\n        model = AutoModel.from_pretrained(str(MODEL_PATHS[model_key]), torch_dtype=torch.bfloat16,\n                                         attn_implementation=\'sdpa\', local_files_only=True).cuda().eval()\n        assert model.config.hidden_size == spec[\'dim\']\n        note(f\'{model_key}: query encoder loaded\')\n\n        @torch.inference_mode()\n        def encode(texts, batch=16):\n            tokenized = tok(texts, truncation=True, max_length=1024)[\'input_ids\']\n            order = np.argsort([-len(row) for row in tokenized], kind=\'stable\')\n            vectors = np.empty((len(texts), spec[\'dim\']), dtype=np.float32)\n            for start in range(0, len(order), batch):\n                index = order[start:start + batch]\n                enc = tok.pad({\'input_ids\': [tokenized[i] for i in index]}, return_tensors=\'pt\').to(\'cuda\')\n                hidden = model(input_ids=enc[\'input_ids\'], attention_mask=enc[\'attention_mask\'], use_cache=False).last_hidden_state\n                mask = enc[\'attention_mask\']\n                if mask[:, -1].sum() == mask.shape[0]:\n                    pooled = hidden[:, -1]\n                else:\n                    lengths = mask.sum(dim=1) - 1\n                    pooled = hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]\n                vectors[index] = F.normalize(pooled.float(), p=2, dim=1).cpu().numpy()\n            return vectors\n\n        vectors = encode([f\'Instruct: {TASK}\\nQuery:{v["text"]}\' for v in VARIANTS])\n        del model, tok\n        torch.cuda.empty_cache()\n    else:\n        cache_hits = sorted(glob.glob(f\'/kaggle/input/**/query_variants_{model_key}.json\', recursive=True))\n        legacy = False\n        if not cache_hits and model_key == \'qwen4b\':\n            cache_path = one_match(\'/kaggle/input/**/query_variants.json\', \'legacy 4B query variants\')\n            vector_path = cache_path.with_name(\'query_variant_vectors.npy\')\n            legacy = True\n        else:\n            assert len(cache_hits) == 1, f\'{model_key}: attach exactly one matching query variant cache, found {cache_hits}\'\n            cache_path = Path(cache_hits[0])\n            vector_path = cache_path.with_name(f\'query_variant_vectors_{model_key}.npy\')\n        saved = json.loads(cache_path.read_text(encoding=\'utf-8\'))\n        vectors = np.load(vector_path, allow_pickle=False)\n        validate_query_cache(saved, vectors, model_key, VARIANTS, TASK, QUERY_IDS, legacy=legacy,\n                             revision=MODEL_REVISIONS[model_key])\n    validate_query_cache(metadata, vectors, model_key, VARIANTS, TASK, QUERY_IDS,\n                         revision=MODEL_REVISIONS[model_key])\n    QUERY_VECTORS[model_key] = vectors\n    np.save(OUT / f\'query_variant_vectors_{model_key}.npy\', vectors)\n    (OUT / f\'query_variants_{model_key}.json\').write_text(json.dumps(metadata, ensure_ascii=False), encoding=\'utf-8\')\n    if list(ACTIVE_MODEL_WEIGHTS) == [\'qwen4b\']:\n        np.save(OUT / \'query_variant_vectors.npy\', vectors)  # legacy output name\n    note(f\'{model_key}: {len(vectors):,} query variants ready, dim={spec["dim"]}\')\nTIMINGS[\'encode_s\'] = round(time.perf_counter() - started, 1)\n\'\'\'\n\nLOAD_VECTORS = r\'\'\'\n# Load/search/release one embedding model at a time, so fusion does not require all three matrices resident.\nREAD_ROWS = 32768\nALL_IDS = None\nTIMINGS[\'load_vectors_s\'] = 0.0\n\n\ndef load_vector_parts(model_key):\n    global ALL_IDS\n    started = time.perf_counter()\n    paths = MODEL_VECTOR_PATHS[model_key]\n    dim = EMBEDDING_SPECS[model_key][\'dim\']\n    parts, ids = [], []\n    read_ids = ALL_IDS is None\n    for key in sorted(paths):\n        t0 = time.perf_counter()\n        array = np.load(paths[key], mmap_mode=\'r\', allow_pickle=False)\n        assert array.dtype == np.float16 and array.shape[1] == dim\n        gpu = torch.empty(array.shape, dtype=torch.float16, device=\'cuda\')\n        for start in range(0, array.shape[0], READ_ROWS):\n            gpu[start:start + READ_ROWS] = torch.from_numpy(np.array(array[start:start + READ_ROWS])).cuda()\n        del array\n        if read_ids:\n            ids_path = Path(paths[key]).with_name(f\'{key}-ids.parquet\')\n            ids.append(pq.read_table(ids_path, columns=[\'chunk_id\'])[\'chunk_id\'].to_numpy())\n        parts.append(gpu)\n        note(f\'{model_key}/{key}: {gpu.shape[0]:,} rows on GPU in {time.perf_counter() - t0:.0f}s\')\n    offsets = np.cumsum([0] + [p.shape[0] for p in parts])\n    if read_ids:\n        ALL_IDS = np.concatenate(ids)\n    assert len(ALL_IDS) == offsets[-1]\n    TIMINGS[\'load_vectors_s\'] += round(time.perf_counter() - started, 1)\n    return parts, offsets\n\'\'\'\n\nSEARCH = r\'\'\'\ntorch.backends.cuda.matmul.allow_tf32 = SEARCH_TF32\nROW_BLOCK, QUERY_BATCH = 262144, 1024\n\n\n@torch.inference_mode()\ndef search(query_vectors, k):\n    k = min(k, len(ALL_IDS))\n    assert k > 0\n    q = torch.from_numpy(np.ascontiguousarray(query_vectors, dtype=np.float32)).cuda()\n    best_s = torch.full((len(q), k), -float(\'inf\'), device=\'cuda\')\n    best_i = torch.full((len(q), k), -1, dtype=torch.int64, device=\'cuda\')\n    for part, offset in zip(PARTS, OFFSETS[:-1]):\n        for row in range(0, part.shape[0], ROW_BLOCK):\n            block = part[row:row + ROW_BLOCK].float()\n            for qs in range(0, len(q), QUERY_BATCH):\n                scores = q[qs:qs + QUERY_BATCH] @ block.T\n                top_s, top_i = torch.topk(scores, min(k, scores.shape[1]), dim=1)\n                cat_s = torch.cat([best_s[qs:qs + QUERY_BATCH], top_s], dim=1)\n                cat_i = torch.cat([best_i[qs:qs + QUERY_BATCH], top_i + int(offset) + row], dim=1)\n                keep_s, order = torch.topk(cat_s, k, dim=1)\n                best_s[qs:qs + QUERY_BATCH] = keep_s\n                best_i[qs:qs + QUERY_BATCH] = torch.gather(cat_i, 1, order)\n            del block\n    result_s, result_i = best_s.cpu().numpy(), best_i.cpu().numpy()\n    assert (result_i >= 0).all() and np.isfinite(result_s).all()\n    return result_s, result_i\n\n\ndef gather_vectors(rows):\n    flat = np.asarray(rows).ravel()\n    part_of = np.searchsorted(OFFSETS, flat, side=\'right\') - 1\n    out = torch.empty((len(flat), DIM), dtype=torch.float32, device=\'cuda\')\n    for p in np.unique(part_of):\n        mask = np.flatnonzero(part_of == p)\n        local = torch.from_numpy(flat[mask] - OFFSETS[p]).cuda()\n        out[torch.from_numpy(mask).cuda()] = PARTS[p].index_select(0, local).float()\n    return out.view(*np.shape(rows), DIM)\n\n\nstarted = time.perf_counter()\nkinds = np.array([v[\'kind\'] for v in VARIANTS])\nowner = np.array([v[\'qi\'] for v in VARIANTS])\nFULL = np.flatnonzero(kinds == \'full\')\nassert (owner[FULL] == np.arange(len(QUERY_TEXT))).all()\nMODEL_RESULTS = {}\nfor model_key in ACTIVE_MODEL_WEIGHTS:\n    t0 = time.perf_counter()\n    DIM = EMBEDDING_SPECS[model_key][\'dim\']\n    PARTS, OFFSETS = load_vector_parts(model_key)\n    variant_vectors = QUERY_VECTORS[model_key]\n    d_scores, d_rows = search(variant_vectors, DENSE_K)\n    subs = [[] for _ in QUERY_TEXT]\n    for i in np.flatnonzero(kinds == \'sub\'):\n        subs[owner[i]].append(ALL_IDS[d_rows[i, :SUB_K]])\n    prf_ids = None\n    if USE_PRF:\n        with torch.inference_mode():\n            feedback = gather_vectors(d_rows[FULL, :PRF_K]).mean(dim=1)\n            q = torch.from_numpy(variant_vectors[FULL]).cuda()\n            prf = torch.nn.functional.normalize((1 - PRF_ALPHA) * q + PRF_ALPHA * feedback, p=2, dim=1).cpu().numpy()\n        _, prf_rows = search(prf, PRF_KEEP)\n        prf_ids = ALL_IDS[prf_rows]\n        del feedback, q, prf, prf_rows\n    MODEL_RESULTS[model_key] = dict(full_ids=ALL_IDS[d_rows[FULL]], full_scores=d_scores[FULL],\n                                   sub_lists=subs, prf_ids=prf_ids)\n    del PARTS, variant_vectors, d_scores, d_rows\n    torch.cuda.empty_cache()\n    note(f\'{model_key}: full/sub/PRF retrieval complete in {time.perf_counter() - t0:.0f}s; GPU vectors released\')\n\nif len(ACTIVE_MODEL_WEIGHTS) == 1:\n    result = MODEL_RESULTS[next(iter(ACTIVE_MODEL_WEIGHTS))]\n    DENSE_FULL_IDS, DENSE_FULL_SCORES = result[\'full_ids\'], result[\'full_scores\']\nelse:\n    # Reference dense ranking for overlap/export; final hybrid fusion adds every model\'s lists once.\n    dense_fused = [rrf([(list(MODEL_RESULTS[key][\'full_ids\'][qi]), weight)\n                        for key, weight in ACTIVE_MODEL_WEIGHTS.items()], RRF_K)[:DENSE_K]\n                   for qi in range(len(QUERY_TEXT))]\n    DENSE_FULL_IDS = np.array([[cid for cid, _ in row] for row in dense_fused], dtype=object)\n    DENSE_FULL_SCORES = np.array([[score for _, score in row] for row in dense_fused], dtype=np.float32)\nTIMINGS[\'search_s\'] = round(time.perf_counter() - started - TIMINGS[\'load_vectors_s\'], 1)\n\'\'\'\n', 'smoke_hybrid_multimodel.py': '"""Kaggle CPU: execute actual notebook load/search/fusion cells with tiny real Torch matrices."""\nimport json\nfrom pathlib import Path\nimport tempfile\nimport time\nfrom unittest.mock import patch\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom hybrid_inputs import EMBEDDING_SPECS, embedding_weights, group_embedding_vectors, validate_hybrid_inputs\nfrom hybrid_retrieval import rrf, model_candidate_lists\nfrom hybrid_embedding_cells import LOAD_VECTORS, SEARCH\n\n\ndef main(fusion_source):\n    import torch\n    original_empty, original_full = torch.empty, torch.full\n\n    def cpu_empty(*args, **kwargs):\n        if kwargs.get(\'device\') == \'cuda\': kwargs[\'device\'] = \'cpu\'\n        return original_empty(*args, **kwargs)\n\n    def cpu_full(*args, **kwargs):\n        if kwargs.get(\'device\') == \'cuda\': kwargs[\'device\'] = \'cpu\'\n        return original_full(*args, **kwargs)\n\n    reports = []\n    with tempfile.TemporaryDirectory() as tmp:\n        root=Path(tmp); corpus=root/\'corpus\'; corpus.mkdir()\n        ids=[\'en0\',\'vi0\',\'vi1\',\'vi2\',\'zh0\',\'zh1\']\n        spans={\'en\':(0,1),\'vi\':(1,4),\'zh\':(4,6)}\n        chunks=[]; all_vectors=[]; original={}\n        for lang,(a,b) in spans.items():\n            p=corpus/f\'chunks/{lang}/part-000.parquet\'; p.parent.mkdir(parents=True)\n            pq.write_table(pa.table({\'chunk_id\':ids[a:b],\'lang\':[lang]*(b-a)}),p);chunks.append(p)\n        manifest=corpus/\'chunk_manifest.json\'\n        manifest.write_text(json.dumps(dict(dataset_version=\'v2\',languages={l:dict(chunks=b-a) for l,(a,b) in spans.items()})))\n        cosines={\'qwen4b\':[.95,.85,.75,.65,.55,.45], \'qwen8b\':[.45,.95,.55,.85,.65,.75], \'octen8b\':[.55,.45,.95,.65,.85,.75]}\n        for key,spec in EMBEDDING_SPECS.items():\n            folder=root/key; (folder/\'vectors\').mkdir(parents=True)\n            matrix=np.zeros((6,spec[\'dim\']),np.float16)\n            matrix[:,0]=cosines[key]; matrix[:,1]=np.sqrt(1-np.array(cosines[key])**2)\n            original[key]=matrix.astype(np.float32)\n            (folder/\'embed_manifest.json\').write_text(json.dumps(dict(complete=True,model=spec[\'model\'],dim=spec[\'dim\'],doc_prefix=spec[\'doc_prefix\'],max_length=1024,dtype=\'torch.bfloat16\')))\n            for lang,(a,b) in spans.items():\n                p=folder/f\'vectors/{lang}-part-000.npy\'; np.save(p,matrix[a:b]);all_vectors.append(p)\n                pq.write_table(pa.table({\'chunk_id\':ids[a:b]}),p.with_name(p.stem+\'-ids.parquet\'))\n\n        cases=[(\'single-4b\',\'qwen4b\',False,{},False),(\'single-qwen8b\',\'qwen8b\',False,{},True),\n               (\'single-octen8b\',\'octen8b\',False,{},True),\n               (\'fusion-three\',\'qwen4b\',True,dict(qwen4b=2,qwen8b=3,octen8b=5),True),\n               (\'fusion-zero-model\',\'qwen4b\',True,dict(qwen4b=2,qwen8b=0,octen8b=5),True)]\n        for name,selected,fusion,weights,variants_enabled in cases:\n            active=embedding_weights(selected,fusion,weights)\n            # In the zero-weight case, its files are not even attached.\n            available=[p for p in all_vectors if p.parent.parent.name in active]\n            grouped=group_embedding_vectors(available,active); paths={}\n            for key in active:\n                _,paths[key],_=validate_hybrid_inputs(chunks,grouped[key],list(spans),manifest,model_key=key)\n            variants=[dict(qi=0,kind=\'full\',text=\'q\')]\n            if variants_enabled: variants.append(dict(qi=0,kind=\'sub\',text=\'sub\'))\n            qvectors={}\n            for key in active:\n                q=np.zeros((len(variants),EMBEDDING_SPECS[key][\'dim\']),np.float32)\n                q[0,0]=1\n                if variants_enabled:q[1,1]=1\n                qvectors[key]=q\n            env=dict(np=np,pq=pq,Path=Path,time=time,torch=torch,rrf=rrf,model_candidate_lists=model_candidate_lists,\n                note=lambda _:None,TIMINGS={},EMBEDDING_SPECS=EMBEDDING_SPECS,ACTIVE_MODEL_WEIGHTS=active,\n                MODEL_VECTOR_PATHS=paths,QUERY_VECTORS=qvectors,VARIANTS=variants,QUERY_TEXT=[\'q\'],\n                DENSE_K=20,SUB_K=2,PRF_K=2,PRF_ALPHA=.3,PRF_KEEP=4,USE_PRF=variants_enabled,\n                USE_QUERY_SPLIT=variants_enabled,SEARCH_TF32=False,RRF_K=60,W_SUB=.5,W_PRF=.5,\n                W_BM25=.3,BM25_ACTIVE=variants_enabled,BM25_IDS=[[\'en0\']] if variants_enabled else [[]],\n                USE_RERANK=False,POOL_MAX=600)\n            with patch.object(torch.Tensor,\'cuda\',lambda self,*a,**kw:self),patch.object(torch,\'empty\',cpu_empty),patch.object(torch,\'full\',cpu_full),patch.object(torch.cuda,\'empty_cache\',lambda:None):\n                exec(LOAD_VECTORS,env); exec(SEARCH,env); exec(fusion_source,env)\n            for key,result in env[\'MODEL_RESULTS\'].items():\n                scores=original[key] @ qvectors[key][0]\n                rank=np.argsort(-scores)\n                assert list(result[\'full_ids\'][0])==[ids[i] for i in rank]\n                np.testing.assert_allclose(result[\'full_scores\'][0],scores[rank],atol=1e-6)\n                if variants_enabled:\n                    subrank=np.argsort(-(original[key] @ qvectors[key][1]))[:2]\n                    assert list(result[\'sub_lists\'][0][0])==[ids[i] for i in subrank]\n                    feedback=original[key][rank[:2]].mean(axis=0)\n                    prf=.7*qvectors[key][0]+.3*feedback;prf/=np.linalg.norm(prf)\n                    expected=np.argsort(-(original[key]@prf))[:4]\n                    assert list(result[\'prf_ids\'][0])==[ids[i] for i in expected]\n            assert env[\'ONLY_DENSE\']==(len(active)==1 and not variants_enabled)\n            assert not any(\'PARTS\'==key for key in env), \'GPU parts retained after retrieval\'\n            reports.append(dict(case=name,active=active,dim={k:EMBEDDING_SPECS[k][\'dim\'] for k in active},dense_sub_prf_match_numpy=True))\n            print(\'PASS\',name,flush=True)\n    report=dict(complete=True,cases=reports,real_torch_cpu=True,gpu_used=False)\n    Path(\'/kaggle/working/hybrid_multimodel_smoke.json\').write_text(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main(Path(\'/kaggle/working/fusion_cell.py.txt\').read_text())\n', 'fusion_cell.py.txt': "\n# One weighted RRF stage across all model full/sub/PRF lists; BM25 appears once.\nONLY_DENSE = len(ACTIVE_MODEL_WEIGHTS) == 1 and not (USE_QUERY_SPLIT or USE_PRF or BM25_ACTIVE)\nFUSED, POOLS = [], []\nfor qi in range(len(QUERY_TEXT)):\n    lists = model_candidate_lists(MODEL_RESULTS, ACTIVE_MODEL_WEIGHTS, qi, W_SUB, W_PRF)\n    if len(BM25_IDS[qi]):\n        lists.append((list(BM25_IDS[qi]), W_BM25))\n    fused = rrf(lists, RRF_K)\n    FUSED.append(fused)\n    POOLS.append(fused[:POOL_MAX] if USE_RERANK else fused)\npool_sizes = [len(p) for p in POOLS]\nprint('fused candidates per query: median', int(np.median([len(f) for f in FUSED])),\n      '| pool: median', int(np.median(pool_sizes)), 'total', sum(pool_sizes))\n"}
from pathlib import Path
import sys,subprocess
for name,source in FILES.items():Path('/kaggle/working',name).write_text(source,encoding='utf-8')
subprocess.run([sys.executable,'-u','/kaggle/working/smoke_hybrid_multimodel.py'],check=True)
